# FreshRoute CE-1 — Data Preparation (Colab-ready)
Course 24CSE0316 | Run top-to-bottom. Upload `freshroute_foodbank_raw.csv` when prompted, or place it at `data/raw/` locally.
Mirrors `scripts/generate_dataset.py` + `scripts/clean_dataset.py`.

In [ ]:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn
import pandas as pd, numpy as np, json, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
sns.set(style='whitegrid')

In [ ]:
# 1) Load raw — Colab upload fallback
try:
    from google.colab import files
    print('Colab detected: upload freshroute_foodbank_raw.csv')
    up = files.upload()
    RAW = list(up.keys())[0]
    df = pd.read_csv(RAW)
except Exception:
    df = pd.read_csv('data/raw/freshroute_foodbank_raw.csv')
print(df.shape, 'dups:', df.duplicated().sum())
print(df.isna().sum()[df.isna().sum()>0])

In [ ]:
# 2) Cleaning: dedup + drop irrelevant + normalise
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop(columns=['donor_contact_number','remarks'])
for c in ['city','state','donor_type','food_type','storage_condition']:
    df[c] = df[c].astype(str).str.strip().str.title()
df['foodbank_id'] = df['foodbank_id'].astype(str).str.strip()
df['transport_available'] = df['transport_available'].astype(str).str.strip().str.title().replace({'Y':'Yes','N':'No'})
df['redistribution_priority'] = df['redistribution_priority'].astype(str).str.strip().str.title().replace({'Med':'Medium'})
df = df[df['redistribution_priority'].isin(['Low','Medium','High'])].reset_index(drop=True)
df['donation_date'] = pd.to_datetime(df['donation_date'], errors='coerce')
df['donation_month'] = df['donation_date'].dt.month
df['donation_dayofweek'] = df['donation_date'].dt.dayofweek
print(df.shape)

In [ ]:
# 3) Fix incorrect -> NaN, then impute (group-wise temp)
num = ['quantity_donated_kg','quantity_available_kg','shelf_life_hours','storage_temp_C','distance_to_foodbank_km','foodbank_capacity_kg','current_stock_kg','beneficiaries_count']
for c in num: df[c] = pd.to_numeric(df[c], errors='coerce')
df.loc[df['quantity_donated_kg']<=0,'quantity_donated_kg'] = np.nan
df.loc[df['distance_to_foodbank_km']<0,'distance_to_foodbank_km'] = np.nan
df.loc[~df['shelf_life_hours'].between(1,240),'shelf_life_hours'] = np.nan
df.loc[~df['storage_temp_C'].between(-5,50),'storage_temp_C'] = np.nan
m = df['quantity_available_kg'].notna() & df['quantity_donated_kg'].notna()
df.loc[m & (df['quantity_available_kg']>df['quantity_donated_kg']),'quantity_available_kg'] = df.loc[m & (df['quantity_available_kg']>df['quantity_donated_kg']),'quantity_donated_kg']
df['storage_temp_C'] = df.groupby('storage_condition')['storage_temp_C'].transform(lambda s: s.fillna(s.median()))
for c in num:
    df[c] = df[c].fillna(df[c].median())
for c in ['shelf_life_hours','beneficiaries_count','foodbank_capacity_kg','current_stock_kg']:
    df[c] = df[c].round(0).astype(int)
df['transport_available'] = df['transport_available'].fillna(df['transport_available'].mode()[0])
print('missing after:', int(df.isna().sum().sum()))

In [ ]:
# 4) Feature engineering + encoding + X/y
df['stock_pressure'] = (df['current_stock_kg']/df['foodbank_capacity_kg']).round(3)
df['surplus_ratio'] = (df['quantity_available_kg']/df['quantity_donated_kg']).round(3)
df['need_per_km'] = (df['beneficiaries_count']/(df['distance_to_foodbank_km']+1)).round(2)
df['is_perishable'] = (df['shelf_life_hours']<24).astype(int)
df['cold_chain_ok'] = ((df['storage_condition']=='Cold')&(df['storage_temp_C']<=8)).astype(int)
cat = ['city','state','donor_type','food_type','storage_condition','foodbank_id','transport_available']
enc = pd.get_dummies(df, columns=cat, drop_first=True, dtype=int)
enc['redistribution_priority'] = LabelEncoder().fit_transform(df['redistribution_priority'])
enc = enc.drop(columns=['donation_id','donation_date'])
X = enc.drop(columns=['redistribution_priority']); y = enc['redistribution_priority']
print('cleaned:', df.shape, '| encoded:', enc.shape, '| X:', X.shape)
print(df['redistribution_priority'].value_counts())

In [ ]:
# 5) EDA (6 graphs for PPT)
df['redistribution_priority'].value_counts().reindex(['Low','Medium','High']).plot(kind='bar', title='Priority dist'); plt.show()
sns.boxplot(data=df, x='food_type', y='quantity_available_kg'); plt.xticks(rotation=20); plt.title('Qty by food type'); plt.show()
sns.boxplot(data=df, x='redistribution_priority', y='shelf_life_hours', order=['Low','Medium','High']); plt.title('Shelf life by priority'); plt.show()
df['donor_type'].value_counts().plot(kind='bar', title='Donor counts'); plt.show()
sns.scatterplot(data=df, x='distance_to_foodbank_km', y='beneficiaries_count', hue='redistribution_priority'); plt.title('Need vs distance'); plt.show()
sns.heatmap(df[['quantity_donated_kg','quantity_available_kg','shelf_life_hours','distance_to_foodbank_km','beneficiaries_count','stock_pressure','need_per_km']].corr(), annot=True, fmt='.2f'); plt.title('Corr'); plt.show()